# Raritan PDU Sandbox

Sandbox notebook to develope code for operating the Raritan PXO power distribution unit (PDU) 
through the Raritan JSON-RPC interface.

The DEMONEXT PDU is a Raritan PXO-2402R-A16, 4 switchable but not individually metered AC outlets,
one metered input.  We have connected a single RH/Temp sensor module (Raritan DX2-T1H1) that will
be used to monitor temperature and humidity in the DEMONEXT electronics box with the PDU.

We used this sandbox to develop and test code that went into the `raritanPDU.py` class. Use that
class for developing standalone programs.

### Sources

 * Raritan Xerxus JSON-RPC API python bindings: https://pypi.org/project/raritan/
 * Documentation: https://help.raritan.com/json-rpc/4.2.20/



In [2]:
import numpy as np
import math
import os
import sys
import time

# Raritan python client

import raritan
from raritan import rpc
from raritan.rpc import pdumodel
from raritan.rpc import peripheral

# we use pathlib for platform-agnostic path generation

from pathlib inport Path

# we use YAML for runtime configuration files

import yaml


### Get local PDU configuration info from the DEMONEXT config file

We will use YAML to format our runtime configuration file so we don't have to hardcode the instrument and 
system setup info and can share a single config file or all elements of the system.

In [ ]:
configDir = "Documents/Config"
configFile = "demonext.cfg"

cfgFile = str(Path.home() / configDir / configFile)

# open config file readonly, get the pdu elements as the pduInfo dictionary

if os.path.exists(cfgFile):
    with open(cfgFile,'r') as stream:
        try:
            config = yaml.safe_load(stream)
        except Exception as exp:
            print(f"ERROR: could not open configuration file {cfgFile}: {exp}")
    
    pduInfo = config['pdu']
else:
    raise ValueError(f"Runtime configuration file {cfgFile} does not exist")

print(pduInfo)

## DEMONEXT Raritan PXO unit

In the OSU lab the Raritan PXO is on the lab net, DHCP, IP address 140.254.79.215
 * admin - PDU admin, restricted
 * telemetry - custom operator account, no admin privs

We connect without certificate verification (`disable_certificate_verification=True`)


In [5]:
pduIP = pduInfo["ipAddr"]
pduUser = pduInfo["userID"]
pduPass = pduInfo["password"]
disableCert = pduInfo["disableCert"]

t0 = time.time()
agent = rpc.Agent("http",pduIP,pduUser,pduPass,disable_certificate_verification=disableCert)
pdu = pdumodel.Pdu("/model/pdu/0",agent)
pdm = peripheral.DeviceManager("/model/peripheraldevicemanager", agent)

dt = time.time()-t0
print(f"Connected to PDU, {dt:.4f} seconds")

Connected to PDU, 0.0030 seconds


### Get the PDU metadata

This gives a snapshot of the PDU capabilities and confirms we have a good link.

In [8]:
t0 = time.time()
metadata = pdu.getMetaData()
dt = time.time() - t0
print(f"PDU metadata query ({dt:.4f} seconds)")
print(metadata)

PDU metadata query (0.5789 seconds)
pdumodel.Pdu.MetaData:
    * nameplate               = pdumodel.Nameplate:
        * manufacturer = "Raritan"
        * brand        = ""
        * model        = "PXO-2402R-A16"
        * partNumber   = ""
        * serialNumber = "1YB3700080"
        * rating       = pdumodel.Nameplate.Rating:
            * voltage   = "100-120V"
            * current   = "12A"
            * frequency = "50/60Hz"
            * power     = "1.2-1.4kVA"
        * imageFileURL = ""
    * ctrlBoardSerial         = "1967300292"
    * hwRevision              = "0x16"
    * fwRevision              = "4.0.20.5-49038"
    * macAddress              = "00:0d:5d:2b:d6:8f"
    * hasSwitchableOutlets    = True
    * hasMeteredOutlets       = False
    * hasLatchingOutletRelays = False
    * isInlineMeter           = False
    * isEnergyPulseSupported  = False
    * hasDCInlets             = False
    * pduOrientation          = pdumodel.Pdu.PduOrientation.PO_NONE


## Read environmental sensor

We have a Raritan DX2-T1H1 temperature/humidity sensor connected on the unit periperhals port:
 * slot 1 is the temperature sensor
 * slot 2 is the humidity sensor

This code shows how to readout the sensors.  The query takes under 1 second most times to return
temperatures. The sensors will be used in the flight electronics box to monitor the box air temperature
and humidity. Typical monitor cadence should be about once per minute.

In [11]:
t0 = time.time()
slots = pdm.getDeviceSlots()
dt = time.time()-t0
print(f"Retrieved peripheral device slots {dt:.4f} seconds")

pduTemp = slots[0].getDevice()
pduRH = slots[1].getDevice()
boxT = pduTemp.device.getReading().value
boxRH = pduRH.device.getReading().value
dt = time.time() - t0
print(f"PDU Sensors: {dt:.4f} seconds")
print(f" Temp = {boxT:.2f} C")
print(f"   RH = {boxRH:.1f}%")


Retrieved peripheral device slots 0.1043 seconds
PDU Sensors: 0.3585 seconds
 Temp = 21.69 C
   RH = 15.7%


## Read inlet power data

Read power data of interest from the inlet sensors.  This PDU model does not have sensors for
monitoring the individual outlets, but we can tell how much power and current are drawn by all 
active devices on the PDU, as well as querying the AC line voltage and current among other data

During lab testing we'll hook up individual devices to the PDU and monitor their power consumption
as part of characterization of the system.

### Read and show inlet metadata

In [14]:
t0 = time.time()
inlets = pdu.getInlets()
md = inlets[0].getMetaData()
dt = time.time() - t0
print(f"PDU inlet metadata ({dt:.4f} seconds):")
print(md)

PDU inlet metadata (0.1248 seconds):
pdumodel.Inlet.MetaData:
    * label              = "I1"
    * plugType           = "IEC 60320 C20"
    * namePlate          = pdumodel.Nameplate:
        * manufacturer = ""
        * brand        = ""
        * model        = ""
        * partNumber   = ""
        * serialNumber = "<not set>"
        * rating       = pdumodel.Nameplate.Rating:
            * voltage   = ""
            * current   = ""
            * frequency = ""
            * power     = ""
        * imageFileURL = ""
    * rating             = pdumodel.Rating:
        * current        = 12
        * decimalCurrent = 12.0
        * minVoltage     = 100
        * maxVoltage     = 120
    * hasWaveformSupport = False
    * isDC               = False


### Inlet sensor data

Data we are interested in are
 * `voltage` = RMS line voltage into the PDU in Volts
 * `lineFrequency` = line frequency in Hz
 * `current` = RMS current drawn in Amps
 * `peakCurrent` = peak RMS current drawn in Amps
 * `activePower` = active power drawn in Watts
 * `apparentPower` = apparent power drawin n VA
 * `activeEnergy` = active energy in Wh

In [17]:
t0 = time.time()
inlets = pdu.getInlets()
sensors = inlets[0].getSensors()
rmsV = sensors.voltage
freq = sensors.lineFrequency
rmsI = sensors.current
peakI = sensors.peakCurrent
watts = sensors.activePower
VA = sensors.apparentPower
kWh = sensors.activeEnergy
print(f"Inlet data query:")
print(f"  RMS Voltage: {rmsV.getReading().value:.2f} V, {freq.getReading().value:.2f} Hz")
print(f"  RMS Current: {rmsI.getReading().value:.3f} A, {peakI.getReading().value:.3f} A peak")
print(f"        Power: {watts.getReading().value:.2f} W")
print(f"               {VA.getReading().value:.1f} VA")
print(f"               {kWh.getReading().value/1000:.2f} kWh")
dt = time.time() - t0
print(f"Query time: {dt:.4f} seconds")

Inlet data query:
  RMS Voltage: 122.45 V, 60.00 Hz
  RMS Current: 0.282 A, 0.425 A peak
        Power: 22.24 W
               36.0 VA
               1.40 kWh
Query time: 0.6628 seconds


## Outlet operations

**BEWARE: the PC is on outlet 1, don't mess with it from the PC!!**

Outlet assignments on the DEMONEXT Raritan PDU are as follows:
 * Outlet 1: DEMONEXT PC
 * Outlet 2: FLI PL23042 CCD camera (includes CFW3-10 filter wheel and Hedrick focuser)
 * Outlet 3: SiTech telescope mount servo controller
 * Outlet 4: unassigned

### Show the settings for each of the 4 outlets

The documentation is opaque at best, this however is useful: https://github.com/tijuca/raritan-pdu-json-rpc-sdk/blob/debian/sid/python-json-rpc-demo.py

In [74]:
OnOff = {True:"ON",False:"OFF"}

print(f"PDU Outlet States:")
outlets = pdu.getOutlets()
for outlet in outlets:
    outSet = outlet.getSettings()
    outSens = outlet.getSensors()
    oss = outSens.outletState
    if outSet.startupState == pdumodel.Outlet.StartupState.SS_ON:
        startup = 'ON'
    else:
        startup = 'OFF'
    print(f"   {outSet.name}: {OnOff[oss.getState().value]}, default = {startup}, cycleDelay: {outSet.cycleDelay} sec")


PDU Outlet States:
   PC: ON, default = ON, cycleDelay: 10 sec
   FLI Camera: OFF, default = OFF, cycleDelay: 10 sec
   SiTech Controller: ON, default = OFF, cycleDelay: 10 sec
   Unassigned: OFF, default = OFF, cycleDelay: 10 sec


### Control outlet on/off state

Use the outlet methods `setPowerState()` and `cyclePowerState()` to power an outlet on and off,
and the power cycle (off, delay, then on).

What we learned from the first round was that the power control functions
`setPowerState()` and `cyclePowerState()` are non-blocking.  

After `setPowerState()` introduce a 1-second `sleep()` to ensure it completes and the following state
query to confirm the new power state.

For `cyclePowerState()` we will need to query the `cycleDelay` setting, and then sleep for
`cycleDelay+2` seconds to ensure it has time to complete before the following state query to
confirm the new power state.

In [76]:
iOut = 1 # numbered 0..3

# setup to operate outlet iOut

t0 = time.time()
outlets = pdu.getOutlets()

outlet = outlets[iOut]

outSet = outlet.getSettings()
outID = outSet.name
cycleDelay = outSet.cycleDelay
outSens = outlet.getSensors()
outState = outSens.outletState
print(f"Done: setup complete, {outID} is {OnOff[outState.getState().value]} ({time.time() - t0:.4f} sec)")

# power off

t0 = time.time()
print(f"Turning {outID} OFF")
outlet.setPowerState(outlet.PowerState.PS_OFF)
time.sleep(1)
print(f"Done: {outID} is {OnOff[outState.getState().value]} ({time.time()-t0:.4f} sec)")

time.sleep(cycleDelay)

# power on

t0 = time.time()
print(f"Turning {outID} ON")
outlet.setPowerState(outlet.PowerState.PS_ON)
time.sleep(1)
print(f"Done: {outID} is {OnOff[outState.getState().value]}... ({time.time()-t0:.4f} sec)")

time.sleep(cycleDelay)

# power cycle: off then on after an internal (10s) cycle delay set in the PDU

t0 = time.time()
print(f"Power cycling {outID}...")
outlet.cyclePowerState()
time.sleep(cycleDelay+1)
print(f"Done: Power cycle complete, {outID} is {OnOff[outState.getState().value]}... ({time.time()-t0:.4f} sec)")

# Note: apparently cyclePowerState() is non-blocking

Done: setup complete, FLI Camera is OFF (0.7394 sec)
Turning FLI Camera OFF
Done: FLI Camera is OFF (1.1388 sec)
Turning FLI Camera ON
Done: FLI Camera is ON... (1.2108 sec)
Power cycling FLI Camera...
Done: Power cycle complete, FLI Camera is ON... (11.1747 sec)


### `powerOn()` Power ON an outlet

Implement as a function that takes 2 arguments:
 * pdu = `pdumodel` instance
 * iOut = outlet to turn on, 0..numOutlets

Returns `True` if outlet powered on, `False` if an invalid outlet number

In [115]:
def powerOn(pdu,iOut):
    numOut = len(pdu.getOutlets())
    # valid outlet?
    if iOut+1 > numOut or iOut < 0:
        print(f"ERROR: invalid outlet {iOut}, must be 0..{numOut-1}")
        return False

    # valid, proceed
    
    outlet = pdu.getOutlets()[iOut]
    outID = outlet.getSettings().name
    outState = outlet.getSensors().outletState

    print(f"Powering {outID} ON")
    outlet.setPowerState(outlet.PowerState.PS_ON)
    time.sleep(1)
    print(f"Done: {outID} is {OnOff[outState.getState().value]}")
    return True

#
iOut = 3
if powerOn(pdu,iOut):
    print(f'powerOn() done')
else:
    print(f'powerOn() returned error')
    

Powering Unassigned ON
Done: Unassigned is ON
powerOn() done


### `powerOff()` Power OFF an outlet

Implement as a function that takes 2 arguments:
 * pdu = `pdumodel` instance
 * iOut = outlet to turn off, 0..numOutlets

Returns `True` if outlet powered off, `False` if an invalid outlet number

In [121]:
def powerOff(pdu,iOut):
    numOut = len(pdu.getOutlets())
    # valid outlet?
    if iOut+1 > numOut or iOut < 0:
        print(f"ERROR: invalid outlet {iOut}, must be 0..{numOut-1}")
        return False

    # valid, proceed
    
    outlet = pdu.getOutlets()[iOut]
    outID = outlet.getSettings().name
    outState = outlet.getSensors().outletState

    print(f"Powering {outID} OFF")
    outlet.setPowerState(outlet.PowerState.PS_OFF)
    time.sleep(1)
    print(f"Done: {outID} is {OnOff[outState.getState().value]}")
    return True

#
iOut = 3
if powerOff(pdu,iOut):
    print(f'powerOff() done')
else:
    print(f'powerOff() returned error')
    

Powering Unassigned OFF
Done: Unassigned is OFF
powerOff() done
